#**SimPy**

SimPy é uma biblioteca em Python para simulação de eventos discretos (DES).

*   Environment — controla o tempo e agenda eventos
*   Process — funções que usam yield para serem "pausadas"
*   Event — timeout, requisição de recurso, liberação
*   Resource — servidores, máquinas, berços, caixas etc.

In [ ]:
!pip install simpy

# **Processo de Chegada e Atendimento**

# Papel do yield
Cada processo é modelado como:
*   yield env.timeout(T) → pausa o processo por T unidades de tempo
*   yield recurso.request() → aguarda recurso livre
*   yield evento → espera evento acontecer


# **Fila M/M/1 com Registro da Ocupação Média**

In [1]:
!pip install simpy

import simpy
import random
import statistics
import math

# ----------------------
# PARÂMETROS DO MODELO
# ----------------------
LAMBDA = 1/5   # taxa de chegada (λ) -> 0.2 clientes / hora
MU = 1/3       # taxa de serviço (μ) -> 0.333 clientes / hora
TEMPO_SIM = 50000  # tempo de simulação

tempos_espera = []
tempos_sistema = []
ocupacao = []
total_clientes = 0

def atendimento(env, nome, servidor):
    global total_clientes
    chegada = env.now
    total_clientes += 1

    with servidor.request() as req:
        yield req

        espera = env.now - chegada
        tempos_espera.append(espera)

        # Tempo de serviço ~ Exp(MU)
        tempo_servico = random.expovariate(MU)
        yield env.timeout(tempo_servico)

        saida = env.now
        tempos_sistema.append(saida - chegada)

def chegadas(env, servidor):
    i = 0
    while True:
        # Intervalo entre chegadas ~ Exp(LAMBDA)
        intervalo = random.expovariate(LAMBDA)
        yield env.timeout(intervalo)
        i += 1
        env.process(atendimento(env, f"Cliente {i}", servidor))

def monitor(env, servidor):
    while True:
        ocupacao.append(servidor.count)
        yield env.timeout(1)

# Ambiente
env = simpy.Environment()
servidor = simpy.Resource(env, capacity=1)
env.process(chegadas(env, servidor))
env.process(monitor(env, servidor))
env.run(until=TEMPO_SIM)

# ----------------------
# TEORIA M/M/1
# ----------------------
def teorico_mm1(lmbd, mu):
    rho = lmbd / mu
    if rho >= 1:
        raise ValueError("Sistema instável: λ >= μ")

    L = rho / (1 - rho)
    Lq = rho**2 / (1 - rho)
    W = 1 / (mu - lmbd)
    Wq = lmbd / (mu * (mu - lmbd))
    P0 = 1 - rho

    return {
        "rho": rho,
        "L": L,
        "Lq": Lq,
        "W": W,
        "Wq": Wq,
        "P0": P0
    }

teo = teorico_mm1(LAMBDA, MU)

print("\n===== RESULTADOS DA SIMULAÇÃO M/M/1 =====")
print(f"Clientes atendidos: {total_clientes}")
print(f"Tempo médio de espera Wq (sim): {statistics.mean(tempos_espera):.4f}")
print(f"Tempo médio no sistema W  (sim): {statistics.mean(tempos_sistema):.4f}")
print(f"Ocupação média ρ (sim):          {statistics.mean(ocupacao):.4f}")

print("\n===== RESULTADOS TEÓRICOS M/M/1 =====")
print(f"ρ   = {teo['rho']:.4f}")
print(f"Lq  = {teo['Lq']:.4f}")
print(f"L   = {teo['L']:.4f}")
print(f"Wq  = {teo['Wq']:.4f}")
print(f"W   = {teo['W']:.4f}")

print("\n===== COMPARAÇÃO DIRETA =====")
print(f"Wq sim ≈ {statistics.mean(tempos_espera):.4f}  | Wq teo = {teo['Wq']:.4f}")
print(f"W  sim ≈ {statistics.mean(tempos_sistema):.4f}  | W  teo = {teo['W']:.4f}")
print(f"ρ  sim ≈ {statistics.mean(ocupacao):.4f}  | ρ  teo = {teo['rho']:.4f}")



===== RESULTADOS DA SIMULAÇÃO M/M/1 =====
Clientes atendidos: 9920
Tempo médio de espera Wq (sim): 4.2152
Tempo médio no sistema W  (sim): 7.1973
Ocupação média ρ (sim):          0.5904

===== RESULTADOS TEÓRICOS M/M/1 =====
ρ   = 0.6000
Lq  = 0.9000
L   = 1.5000
Wq  = 4.5000
W   = 7.5000

===== COMPARAÇÃO DIRETA =====
Wq sim ≈ 4.2152  | Wq teo = 4.5000
W  sim ≈ 7.1973  | W  teo = 7.5000
ρ  sim ≈ 0.5904  | ρ  teo = 0.6000


# **Fila M/M/1 com Calendário de Funcionamento (Paradas Diárias)**

In [5]:
!pip install simpy

import simpy
import random
import statistics

# ----------------------
# PARÂMETROS DO MODELO
# ----------------------
LAMBDA = 1/5       # taxa de chegada (λ)
MU = 1/3           # taxa de serviço (μ)
TEMPO_SIM = 50000  # tempo total de simulação

HORAS_DIA = 24
HORAS_PARADA = 4
HORA_INICIO_PARADA = 2  # 02h-06h

# ----------------------
# ESTATÍSTICAS
# ----------------------
tempos_espera = []
tempos_sistema = []
ocupacao = []
total_clientes = 0

# ----------------------
# CONTROLE DA MÁQUINA
# ----------------------
maquina_ativa = True

# ----------------------
# PROCESSO DO CLIENTE
# ----------------------
def cliente(env, nome, servidor):
    global total_clientes
    chegada = env.now
    total_clientes += 1

    with servidor.request() as req:
        yield req

        # Espera máquina ligar (durante paradas)
        while not maquina_ativa:
            yield env.timeout(0.5)

        espera = env.now - chegada
        tempos_espera.append(espera)

        # Serviço exponencial
        tempo_servico = random.expovariate(MU)
        yield env.timeout(tempo_servico)

        tempos_sistema.append(env.now - chegada)


# ----------------------
# CHEGADAS POISSON
# ----------------------
def chegadas(env, servidor):
    i = 0
    while True:
        yield env.timeout(random.expovariate(LAMBDA))
        i += 1
        env.process(cliente(env, f"Cliente {i}", servidor))


# ----------------------
# MONITORAMENTO
# ----------------------
def monitor(env, servidor):
    while True:
        ocupacao.append(servidor.count)
        yield env.timeout(1)


# ----------------------
# CALENDÁRIO DA MÁQUINA (CORRETO)
# ----------------------
def calendario_maquina(env):
    global maquina_ativa

    while True:
        hora = env.now % HORAS_DIA

        # Máquina deve estar desligada?
        if HORA_INICIO_PARADA <= hora < HORA_INICIO_PARADA + HORAS_PARADA:
            maquina_ativa = False
        else:
            maquina_ativa = True

        yield env.timeout(1)


# ----------------------
# EXECUÇÃO DA SIMULAÇÃO
# ----------------------
env = simpy.Environment()
servidor = simpy.Resource(env, capacity=1)

env.process(chegadas(env, servidor))
env.process(monitor(env, servidor))
env.process(calendario_maquina(env))

env.run(until=TEMPO_SIM)

# ----------------------
# RESULTADOS
# ----------------------
print("\n===== RESULTADOS DA SIMULAÇÃO (M/M/1 COM PARADAS DIÁRIAS) =====")
print(f"Clientes atendidos: {total_clientes}")
print(f"Tempo médio de espera Wq: {statistics.mean(tempos_espera):.4f}")
print(f"Tempo médio no sistema W: {statistics.mean(tempos_sistema):.4f}")
print(f"Ocupação média ρ (simulada): {statistics.mean(ocupacao):.4f}")



===== RESULTADOS DA SIMULAÇÃO (M/M/1 COM PARADAS DIÁRIAS) =====
Clientes atendidos: 9977
Tempo médio de espera Wq: 6.3444
Tempo médio no sistema W: 9.3287
Ocupação média ρ (simulada): 0.6590


# **Fila M/M/1 com Calendário de Funcionamento (Atualização Automática)**

In [6]:
!pip install simpy

import simpy
import random
import statistics

# ----------------------
# PARÂMETROS DO MODELO
# ----------------------
LAMBDA = 1/5       # taxa de chegada (λ)
MU = 1/3           # taxa de serviço (μ)
TEMPO_SIM = 50000  # tempo total de simulação

HORAS_DIA = 24
HORAS_PARADA = 4
HORA_INICIO_PARADA = 2  # 02h-06h

# ----------------------
# ESTATÍSTICAS
# ----------------------
tempos_espera = []
tempos_sistema = []
ocupacao = []
total_clientes = 0

# ----------------------
# EVENTO DE MÁQUINA LIGADA
# ----------------------
maquina_ligada = None

# ----------------------
# PROCESSO DO CLIENTE
# ----------------------
def atendimento(env, nome, servidor):
    global total_clientes, maquina_ligada

    chegada = env.now
    total_clientes += 1

    with servidor.request() as req:
        # Espera o recurso estar disponível
        yield req

        # Espera máquina estar ligada
        yield maquina_ligada

        espera = env.now - chegada
        tempos_espera.append(espera)

        # Serviço exponencial
        tempo_servico = random.expovariate(MU)
        yield env.timeout(tempo_servico)

        tempos_sistema.append(env.now - chegada)


# ----------------------
# GERADOR DE CHEGADAS (POISSON)
# ----------------------
def chegadas(env, servidor):
    i = 0
    while True:
        yield env.timeout(random.expovariate(LAMBDA))
        i += 1
        env.process(atendimento(env, f"Cliente {i}", servidor))

# ----------------------
# MONITORAMENTO (OCUPAÇÃO)
# ----------------------
def monitor(env, servidor):
    while True:
        ocupacao.append(servidor.count)
        yield env.timeout(1)

# ----------------------
# CALENDÁRIO DA MÁQUINA (USANDO EVENTOS)
# ----------------------
def calendario_maquina(env):
    global maquina_ligada

    while True:
        hora_atual = env.now % HORAS_DIA

        # Se estamos FORA do período de parada
        if hora_atual < HORA_INICIO_PARADA or hora_atual >= HORA_INICIO_PARADA + HORAS_PARADA:

            # Cria evento e liga máquina imediatamente
            maquina_ligada = env.event()
            maquina_ligada.succeed()

            # Tempo até início da próxima parada
            if hora_atual < HORA_INICIO_PARADA:
                tempo_ate_parada = HORA_INICIO_PARADA - hora_atual
            else:
                tempo_ate_parada = HORAS_DIA - hora_atual + HORA_INICIO_PARADA

            yield env.timeout(tempo_ate_parada)

        else:
            # Estamos dentro da janela de parada — máquina DESLIGADA
            maquina_ligada = env.event()  # ainda não 'succeed'

            # Tempo restante até ligar
            tempo_restante = (HORA_INICIO_PARADA + HORAS_PARADA) - hora_atual

            yield env.timeout(tempo_restante)

            # Ao terminar a parada, máquina RELIGA
            maquina_ligada.succeed()


# ----------------------
# EXECUÇÃO DA SIMULAÇÃO
# ----------------------
env = simpy.Environment()
servidor = simpy.Resource(env, capacity=1)

# Inicia processos
env.process(chegadas(env, servidor))
env.process(monitor(env, servidor))
env.process(calendario_maquina(env))

env.run(until=TEMPO_SIM)

# ----------------------
# RESULTADOS
# ----------------------
print("\n===== RESULTADOS DA SIMULAÇÃO (M/M/1 COM PARADAS DIÁRIAS) =====")
print(f"Clientes atendidos: {total_clientes}")
print(f"Tempo médio de espera Wq: {statistics.mean(tempos_espera):.4f}")
print(f"Tempo médio no sistema W: {statistics.mean(tempos_sistema):.4f}")
print(f"Ocupação média ρ (simulada): {statistics.mean(ocupacao):.4f}")



===== RESULTADOS DA SIMULAÇÃO (M/M/1 COM PARADAS DIÁRIAS) =====
Clientes atendidos: 10043
Tempo médio de espera Wq: 5.9793
Tempo médio no sistema W: 8.9412
Ocupação média ρ (simulada): 0.6515


# **Porto com 2 berços**

O processo de chegada é aleatório, sendo regido por uma distribuição exponencial negativa com média 1/6. O processo de atendimento é aleatório, sendo regido por uma distribuição exponencial negativa com média 1/7.5. Para atravessar o canal do porto leva-se um tempo regido por uma distribuição normal com média de 1 hora e desvio padrão 10% da média.

In [9]:
!pip install simpy

import simpy
import random
import statistics

# ==========================
# PARÂMETROS
# ==========================

# Chegadas
LAMBDA = 1/6         # média 6 horas → taxa = 1/6

# Travessia do canal (entrada e saída)
CANAL_MEAN = 1.0     # 60 min = 1 h
CANAL_SD = 0.1       # 6 min = 0.1 h

# Serviço nos berços
MU = 1/7.5           # média 7,5 horas → taxa = 1/7.5

# Quantidade de berços
NUM_BERCOS = 2

# Tempo total
TEMPO_SIM = 24*365

# ==========================
# ESTATÍSTICAS
# ==========================
tempos_sistema = []
tempos_fila = []
total_navios = 0

# ==========================
# PROCESSO DO NAVIO
# ==========================
def atendimento(env, nome, bercos):
    global total_navios

    chegada = env.now
    total_navios += 1

    # --- Travessia do canal de entrada ---
    tempo_entrada = max(0, random.normalvariate(CANAL_MEAN, CANAL_SD))
    yield env.timeout(tempo_entrada)

    # --- Fila nos berços ---
    with bercos.request() as req:
        yield req

        tempo_fila = env.now - chegada
        tempos_fila.append(tempo_fila)

        # --- Serviço no berço ---
        tempo_servico = random.expovariate(MU)
        yield env.timeout(tempo_servico)

    # --- Travessia do canal de saída ---
    tempo_saida = max(0, random.normalvariate(CANAL_MEAN, CANAL_SD))
    yield env.timeout(tempo_saida)

    tempos_sistema.append(env.now - chegada)

# ==========================
# GERADOR DE CHEGADAS
# ==========================
def chegadas(env, bercos):
    i = 0
    while True:
        # Chegadas Poisson
        yield env.timeout(random.expovariate(LAMBDA))
        i += 1
        env.process(atendimento(env, f"Navio {i}", bercos))

# ==========================
# EXECUÇÃO
# ==========================
env = simpy.Environment()
bercos = simpy.Resource(env, capacity=NUM_BERCOS)
env.process(chegadas(env, bercos))
env.run(until=TEMPO_SIM)

# ==========================
# RESULTADOS
# ==========================
print("\n===== RESULTADOS DO MODELO DE NAVIOS =====")
print(f"Navios processados: {total_navios}")
print(f"Tempo médio na fila: {statistics.mean(tempos_fila):.2f} horas")
print(f"Tempo médio no sistema: {statistics.mean(tempos_sistema):.2f} horas")



===== RESULTADOS DO MODELO DE NAVIOS =====
Navios processados: 1428
Tempo médio na fila: 4.62 horas
Tempo médio no sistema: 12.79 horas


# **Porto com 2 berços; um navio por vez no canal de acesso**
**Lógica 1: Requisição sequencial: canal -> berço**

In [10]:
!pip install simpy

import simpy
import random
import statistics

# ==========================
# PARÂMETROS
# ==========================
# Chegadas
LAMBDA = 1/6     # média 6 horas → taxa = 1/6

# Travessia do canal
CANAL_MEAN = 1.0     # 60 min = 1 h
CANAL_SD = 0.1       # 6 min = 0.1 h

# Serviço nos berços
MU = 1/7.5           # média 7,5 horas → taxa = 1/7.5

# Quantidade de berços
NUM_BERCOS = 2

# Canal com capacidade 1
CAPACIDADE_CANAL = 1

# Tempo total
TEMPO_SIM = 24*365

# ==========================
# ESTATÍSTICAS
# ==========================
tempos_sistema = []
tempos_fila_berco = []
tempos_fila_canal_entrada = []
tempos_fila_canal_saida = []
total_navios = 0

# ==========================
# PROCESSO DO NAVIO
# ==========================
def atendimento(env, nome, bercos, canal):
    global total_navios

    chegada = env.now
    total_navios += 1

    # --- CANAL DE ENTRADA (somente 1 navio por vez) ---
    canal_chegada = env.now
    with canal.request() as req_canal:
        yield req_canal
        tempos_fila_canal_entrada.append(env.now - canal_chegada)

        tempo_entrada = max(0, random.normalvariate(CANAL_MEAN, CANAL_SD))
        yield env.timeout(tempo_entrada)

    # --- FILA DOS BERÇOS ---
    comeco_fila = env.now
    with bercos.request() as req:
        yield req
        tempos_fila_berco.append(env.now - comeco_fila)

        # --- SERVIÇO NO BERÇO ---
        tempo_servico = random.expovariate(MU)
        yield env.timeout(tempo_servico)

    # --- CANAL DE SAÍDA (também 1 navio por vez) ---
    canal_saida = env.now
    with canal.request() as req_canal_saida:
        yield req_canal_saida
        tempos_fila_canal_saida.append(env.now - canal_saida)

        tempo_saida = max(0, random.normalvariate(CANAL_MEAN, CANAL_SD))
        yield env.timeout(tempo_saida)

    # --- TEMPO TOTAL ---
    tempos_sistema.append(env.now - chegada)

# ==========================
# GERADOR DE CHEGADAS
# ==========================
def chegadas(env, bercos, canal):
    i = 0
    while True:
        yield env.timeout(random.expovariate(LAMBDA))
        i += 1
        env.process(atendimento(env, f"Navio {i}", bercos, canal))

# ==========================
# EXECUÇÃO
# ==========================
env = simpy.Environment()
bercos = simpy.Resource(env, capacity=NUM_BERCOS)
canal = simpy.Resource(env, capacity=1)
env.process(chegadas(env, bercos, canal))
env.run(until=TEMPO_SIM)

# ==========================
# RESULTADOS
# ==========================
print("\n===== RESULTADOS DO MODELO DE NAVIOS (CANAL EXCLUSIVO) =====")
print(f"Navios processados: {total_navios}")
print(f"Tempo médio no sistema: {statistics.mean(tempos_sistema):.2f} horas")
print(f"Tempo médio na fila do canal (entrada): {statistics.mean(tempos_fila_canal_entrada):.2f} horas")
print(f"Tempo médio na fila do canal (saída): {statistics.mean(tempos_fila_canal_saida):.2f} horas")
print(f"Tempo médio na fila do berço: {statistics.mean(tempos_fila_berco):.2f} horas")



===== RESULTADOS DO MODELO DE NAVIOS (CANAL EXCLUSIVO) =====
Navios processados: 1365
Tempo médio no sistema: 12.77 horas
Tempo médio na fila do canal (entrada): 0.23 horas
Tempo médio na fila do canal (saída): 0.27 horas
Tempo médio na fila do berço: 3.10 horas


# **Porto com 2 berços; um navio por vez no canal de acesso**
**Lógica 2: Requisição sequencial: berço -> canal**

In [17]:
!pip install simpy

import simpy
import random
import statistics

# ==========================
# PARÂMETROS
# ==========================
# Chegadas (exponencial média 6h)
LAMBDA = 1/6

# Travessia do canal (Normal 1h, desvio 0.1h)
CANAL_MEAN = 1.0
CANAL_SD = 0.1

# Serviço no berço (exponencial média 7,5h)
MU = 1/7.5

# Quantidade de berços
NUM_BERCOS = 2

# Canal exclusivo
CAPACIDADE_CANAL = 1

# Tempo de simulação
TEMPO_SIM = 24*365

# ==========================
# ESTATÍSTICAS
# ==========================
tempos_sistema = []
fila_berco = []
fila_canal_entrada = []
fila_canal_saida = []
total_navios = 0

# ==========================
# PROCESSO DO NAVIO
# ==========================
def atendimento(env, nome, bercos, canal):
    global total_navios
    chegada = env.now
    total_navios += 1

    # -- NAVIO PEDE O BERÇO PRIMEIRO ---
    inicio_fila_berco = env.now
    with bercos.request() as req_berco:
        yield req_berco
        fila_berco.append(env.now - inicio_fila_berco)

        # --- SÓ AGORA ELE PEDE O CANAL DE ENTRADA ---
        inicio_fila_canal_ent = env.now
        with canal.request() as req_canal_ent:
            yield req_canal_ent
            fila_canal_entrada.append(env.now - inicio_fila_canal_ent)
            # Atravessa canal de ENTRADA
            tempo_ent = max(0, random.normalvariate(CANAL_MEAN, CANAL_SD))
            yield env.timeout(tempo_ent)

        # --- AGORA OCORRE O SERVIÇO NO BERÇO ---
        tempo_servico = random.expovariate(MU)
        yield env.timeout(tempo_servico)

        # --- NAVIO PEDE CANAL PARA SAIR ---
        inicio_fila_canal_sai = env.now
        with canal.request() as req_canal_sai:
            yield req_canal_sai
            fila_canal_saida.append(env.now - inicio_fila_canal_sai)

            # Travessia do canal de SAÍDA
            tempo_sai = max(0, random.normalvariate(CANAL_MEAN, CANAL_SD))
            yield env.timeout(tempo_sai)

    # --- TEMPO TOTAL ---
    tempos_sistema.append(env.now - chegada)

# ==========================
# GERADOR DE CHEGADAS
# ==========================
def chegadas(env, bercos, canal):
    i = 0
    while True:
        yield env.timeout(random.expovariate(LAMBDA))
        i += 1
        env.process(atendimento(env, f"Navio {i}", bercos, canal))

# ==========================
# EXECUÇÃO
# ==========================
env = simpy.Environment()
bercos = simpy.Resource(env, capacity=NUM_BERCOS)
canal = simpy.Resource(env, capacity=1)
env.process(chegadas(env, bercos, canal))
env.run(until=TEMPO_SIM)

# ==========================
# RESULTADOS
# ==========================
print("\n===== RESULTADOS — NAVIO PEDE BERÇO PRIMEIRO =====")
print(f"Navios processados: {total_navios}")
print(f"Tempo médio na fila do berço: {statistics.mean(fila_berco) if fila_berco else 0:.2f} h")
print(f"Tempo médio na fila do canal (entrada): {statistics.mean(fila_canal_entrada) if fila_canal_entrada else 0:.2f} h")
print(f"Tempo médio na fila do canal (saída): {statistics.mean(fila_canal_saida) if fila_canal_saida else 0:.2f} h")
print(f"Tempo médio total no sistema: {statistics.mean(tempos_sistema) if tempos_sistema else 0:.2f} h")


===== RESULTADOS — NAVIO PEDE BERÇO PRIMEIRO =====
Navios processados: 1414
Tempo médio na fila do berço: 9.59 h
Tempo médio na fila do canal (entrada): 0.14 h
Tempo médio na fila do canal (saída): 0.09 h
Tempo médio total no sistema: 19.27 h


# **Atendimento de Navios Porta Contêineres (Cabotagem)**

In [20]:
!pip install simpy

import simpy
import random
import statistics

# ==========================
# PARÂMETROS
# ==========================

# Chegadas (exponencial com média de 12h)
LAMBDA = 1/12    # taxa = 1/12 → média 12 horas

# Travessia do canal (Normal 1h, desvio 0.1h)
CANAL_MEAN = 1.0     # horas (60 min)
CANAL_SD = 0.1       # horas (6 min)

# Serviço no berço: depende dos TEUs e da triangular em minutos
TEUS_TIPO1 = 3500
TEUS_TIPO2 = 5000
PERC_MOVIMENTADO = 0.20  # 20%

# Distribuição de tipos
P_TIPO1 = 0.8  # 80%
P_TIPO2 = 0.2  # 20%

# Berços (dois berços separados)
CAP_BERCO1 = 1
CAP_BERCO2 = 1

# Canal exclusivo
CAPACIDADE_CANAL = 1

# Tempo de simulação (horas)
TEMPO_SIM = 24 * 365  # 1 ano

# ==========================
# ESTATÍSTICAS
# ==========================

# Tempos médios por tipo
fila_berco_tipo1 = []
fila_berco_tipo2 = []

tempo_sistema_tipo1 = []
tempo_sistema_tipo2 = []

# (Opcional) Para ver também canal, se quiser
fila_canal_entrada_tipo1 = []
fila_canal_entrada_tipo2 = []

fila_canal_saida_tipo1 = []
fila_canal_saida_tipo2 = []

total_navios = 0
total_tipo1 = 0
total_tipo2 = 0


# ==========================
# FUNÇÕES AUXILIARES
# ==========================
def tempo_servico_berco(teus):
    """
    Calcula tempo de serviço no berço com base em:
    - 20% dos TEUs movimentados
    - tempo por movimentação ~ triangular(1.0, 1.5, 1.8) minutos

    Retorna tempo em HORAS.
    """
    movimentos = PERC_MOVIMENTADO * teus
    tempo_por_movimento_min = random.triangular(0.666, 1.2, 0.90)  # min
    tempo_total_min = movimentos * tempo_por_movimento_min
    return tempo_total_min / 60.0  # converte para horas


# ==========================
# PROCESSO DO NAVIO
# ==========================
def atendimento(env, nome, berco1, berco2, canal):
    global total_navios, total_tipo1, total_tipo2

    chegada = env.now
    total_navios += 1

    # Sorteia tipo de navio (1 ou 2)
    if random.random() < P_TIPO1:
        tipo = 1
        total_tipo1 += 1
        teus = TEUS_TIPO1
    else:
        tipo = 2
        total_tipo2 += 1
        teus = TEUS_TIPO2

    # ==========================
    # 1️ - ESCOLHA / RESTRIÇÃO DO BERÇO
    # ==========================
    # Tipo 1 pode ir para berço 1 ou 2 → escolhe o que tiver fila menor
    if tipo == 1:
        fila_1 = berco1.count + len(berco1.queue)
        fila_2 = berco2.count + len(berco2.queue)
        if fila_1 <= fila_2:
            berco_alvo = berco1
        else:
            berco_alvo = berco2
    else:
        # Tipo 2 só pode usar o berço 2 (restrição de calado)
        berco_alvo = berco2

    # ==========================
    # 2️ - FILA DO BERÇO
    # ==========================
    inicio_fila_berco = env.now
    with berco_alvo.request() as req_berco:
        yield req_berco
        tempo_espera_berco = env.now - inicio_fila_berco

        if tipo == 1:
            fila_berco_tipo1.append(tempo_espera_berco)
        else:
            fila_berco_tipo2.append(tempo_espera_berco)

        # ==========================
        # 3️ - DEPOIS DO BERÇO, PEDIR CANAL DE ENTRADA
        # ==========================
        inicio_fila_canal_ent = env.now
        with canal.request() as req_canal_ent:
            yield req_canal_ent
            tempo_fila_canal_ent = env.now - inicio_fila_canal_ent

            if tipo == 1:
                fila_canal_entrada_tipo1.append(tempo_fila_canal_ent)
            else:
                fila_canal_entrada_tipo2.append(tempo_fila_canal_ent)

            # Travessia de ENTRADA
            tempo_ent = max(0, random.normalvariate(CANAL_MEAN, CANAL_SD))
            yield env.timeout(tempo_ent)

        # ==========================
        # 4️ - SERVIÇO NO BERÇO (MOVIMENTAÇÃO DE CONTÊINERES)
        # ==========================
        tempo_servico = tempo_servico_berco(teus)
        yield env.timeout(tempo_servico)

        # ==========================
        # 5️ - CANAL DE SAÍDA
        # ==========================
        inicio_fila_canal_sai = env.now
        with canal.request() as req_canal_sai:
            yield req_canal_sai
            tempo_fila_canal_sai = env.now - inicio_fila_canal_sai

            if tipo == 1:
                fila_canal_saida_tipo1.append(tempo_fila_canal_sai)
            else:
                fila_canal_saida_tipo2.append(tempo_fila_canal_sai)

            tempo_sai = max(0, random.normalvariate(CANAL_MEAN, CANAL_SD))
            yield env.timeout(tempo_sai)

    # ==========================
    # 6️ - TEMPO TOTAL NO TERMINAL
    # ==========================
    tempo_total = env.now - chegada
    if tipo == 1:
        tempo_sistema_tipo1.append(tempo_total)
    else:
        tempo_sistema_tipo2.append(tempo_total)

# ==========================
# GERADOR DE CHEGADAS
# ==========================
def chegadas(env, berco1, berco2, canal):
    i = 0
    while True:
        # Processo Poisson (exponencial entre chegadas)
        yield env.timeout(random.expovariate(LAMBDA))
        i += 1
        env.process(atendimento(env, f"Navio {i}", berco1, berco2, canal))

# ==========================
# EXECUÇÃO
# ==========================
env = simpy.Environment()
berco1 = simpy.Resource(env, capacity=CAP_BERCO1)
berco2 = simpy.Resource(env, capacity=CAP_BERCO2)
canal = simpy.Resource(env, capacity=CAPACIDADE_CANAL)
env.process(chegadas(env, berco1, berco2, canal))
env.run(until=TEMPO_SIM)

# ==========================
# RESULTADOS
# ==========================
print("\n===== RESULTADOS — DOIS TIPOS DE NAVIO =====")
print(f"Navios processados: {total_navios}")
print(f"  Tipo 1: {total_tipo1}")
print(f"  Tipo 2: {total_tipo2}")

def media(v):
    return statistics.mean(v) if v else 0.0

print("\n--- TEMPO MÉDIO EM FILA DE BERÇO ---")
print(f"Tipo 1: {media(fila_berco_tipo1):.2f} h")
print(f"Tipo 2: {media(fila_berco_tipo2):.2f} h")

print("\n--- TEMPO MÉDIO NO TERMINAL ---")
print(f"Tipo 1: {media(tempo_sistema_tipo1):.2f} h")
print(f"Tipo 2: {media(tempo_sistema_tipo2):.2f} h")



===== RESULTADOS — DOIS TIPOS DE NAVIO =====
Navios processados: 693
  Tipo 1: 573
  Tipo 2: 120

--- TEMPO MÉDIO EM FILA DE BERÇO ---
Tipo 1: 3.36 h
Tipo 2: 3.93 h

--- TEMPO MÉDIO NO TERMINAL ---
Tipo 1: 16.27 h
Tipo 2: 21.40 h


# **Carregamento de um navio com soja**

In [9]:
!pip install simpy

import simpy
import random

# =======================================================
# PARÂMETROS DO MODELO
# =======================================================
CAPACIDADE_SILO = 100_000      # toneladas de soja disponíveis
TAXA_CARREGAMENTO = 2_000      # t/h do carregador (shiploader)
CARGA_NAVIO = 50_000           # t que o navio precisa carregar
TEMPO_ATRACACAO = 2            # h para atracar e preparar o carregamento

# =======================================================
# PROCESSO: SILO
# =======================================================
class Silo:
    def __init__(self, env, capacidade_inicial):
        self.env = env
        self.capacidade = capacidade_inicial
        self.monitoramento = []

    def retirar(self, quantidade):
        """Retira soja do silo, respeitando limite."""
        retirada_real = min(quantidade, self.capacidade)
        self.capacidade -= retirada_real
        return retirada_real

# =======================================================
# PROCESSO: CARREGAMENTO DO NAVIO
# =======================================================
def carregar_navio(env, nome, berco, silo):
    chegada = env.now
    print(f"{env.now:.2f} h - {nome} chegou ao porto.")

    # 1. Atracar no berço
    with berco.request() as req:
        yield req
        print(f"{env.now:.2f} h - {nome} iniciou atracação.")
        yield env.timeout(TEMPO_ATRACACAO)

        print(f"{env.now:.2f} h - {nome} começou o carregamento.")

        carga_restante = CARGA_NAVIO

        # 2. Processo de carregamento contínuo
        while carga_restante > 0 and silo.capacidade > 0:
            # Shiploader tenta carregar em 1 hora
            yield env.timeout(1)

            quantidade = silo.retirar(TAXA_CARREGAMENTO)
            carga_restante -= quantidade

            print(f"{env.now:.2f} h - {nome} carregou {quantidade:.0f} t "
                  f"(restam {max(carga_restante, 0):.0f} t para completar).")

        if carga_restante > 0:
            print(f"{env.now:.2f} h - {nome} saiu INCOMPLETO! Faltaram {carga_restante:.0f} t.")
        else:
            print(f"{env.now:.2f} h - {nome} carregou COMPLETO e desatracou.")

    tempo_total = env.now - chegada
    print(f"==> Tempo total do {nome}: {tempo_total:.2f} h\n")

# =======================================================
# SIMULAÇÃO
# =======================================================
env = simpy.Environment()
berco = simpy.Resource(env, capacity=1)
silo = Silo(env, CAPACIDADE_SILO)
env.process(carregar_navio(env, "Navio Graneleiro", berco, silo))
env.run(until=200)



0.00 h - Navio Graneleiro chegou ao porto.
0.00 h - Navio Graneleiro iniciou atracação.
2.00 h - Navio Graneleiro começou o carregamento.
3.00 h - Navio Graneleiro carregou 2000 t (restam 48000 t para completar).
4.00 h - Navio Graneleiro carregou 2000 t (restam 46000 t para completar).
5.00 h - Navio Graneleiro carregou 2000 t (restam 44000 t para completar).
6.00 h - Navio Graneleiro carregou 2000 t (restam 42000 t para completar).
7.00 h - Navio Graneleiro carregou 2000 t (restam 40000 t para completar).
8.00 h - Navio Graneleiro carregou 2000 t (restam 38000 t para completar).
9.00 h - Navio Graneleiro carregou 2000 t (restam 36000 t para completar).
10.00 h - Navio Graneleiro carregou 2000 t (restam 34000 t para completar).
11.00 h - Navio Graneleiro carregou 2000 t (restam 32000 t para completar).
12.00 h - Navio Graneleiro carregou 2000 t (restam 30000 t para completar).
13.00 h - Navio Graneleiro carregou 2000 t (restam 28000 t para completar).
14.00 h - Navio Graneleiro carreg